In [5]:
import duckdb
conn = duckdb.connect()

conn.execute("INSTALL mysql; LOAD mysql;")
conn.execute("ATTACH 'host=localhost user=root password=your_password database=ra_db_2024' AS mysql_db (TYPE MYSQL);")

# Save query result to CSV
conn.execute("""
    COPY (SELECT * FROM mysql_db.ra_db_2024.trainings where country = 'Somalia') 
    TO 'trainings.csv' (FORMAT CSV, HEADER)
""")

In [69]:
#read csv to df
import pandas as pd

df = pd.read_csv('/Users/victor/Documents/projects/mini_projects/regreen_cleaning/2026/ra_v1/somalia/trainings.csv')
df.head()

,id,uploaded_at,module,enumerator_name,date,survey_name,country,region,district_commune_woreda,training_topic,training_type,training_date,training_venue,partners,total_participants,male_participants,female_participants,youth_participants,notes
0,530,2020-09-25 07:11:57,Training,Ahmed,28/4/2020,Regreening Africa,Somalia,PUNTLAND,Bari Ufeyn,Good Nursery Management Practice,"practical, theory",28/4/2020,On Farm/Nursery,CARE International/Ministry of Environment of ...,5,4,1,2,Improve Nursery management by the operators
1,531,2020-09-25 07:11:57,Training,Ahmed,29/4/2020,Regreening Africa,Somalia,Puntland State,Bari /Bosaso,Good Nursery Management Practice,"Practical, theory",27/4/2020,Nursery Garden,CARE International / Ministry of Environment &...,5,3,2,2,Practice the training knowledge
2,532,2020-09-25 07:11:57,Training,Ahmed,4/5/2020,Regreening Africa,Somalia,Puntland State of Somalia,"Qardho, Bari",Good Nursery Management Practice,"Practical, theory",2/5/2020,Ministry of Environment,CARE International & Ministry of Environment,5,4,1,1,Utilize the knowledge gained in the nurseries
3,864,2020-09-25 07:11:57,Training,mohamed ahmed,27/7/2020,Regreening Africa,Somalia,somaliland,odwayne,nursery management training,practical and theory,22/7/2020,ceelsame nursery,Abdirahman Saleeban,9,5,4,3,NaN
4,865,2020-09-25 07:11:57,Training,mohamed ahmed Mohammed,28/7/2020,Regreening Africa,Somalia,somaliland,Odwayne,nursery group training with Somali translated ...,theory and practical,24/7/2020,Qaloocato nursery,Abdirahman Saleeban,10,8,2,0,NaN


In [81]:
import copy
#read json to memory
import json
import re

with open('/Users/victor/Documents/projects/mini_projects/regreen_cleaning/2026/ra_v1/somalia/engagements_template.json', 'r') as f:
    engagements_template = json.load(f)

#empty json list
data_jsons = []

# Create a list of engagement JSON templates
json_list = []
loop_size = len(df.index)
for i in range(loop_size):
    json_list.append(copy.deepcopy(engagements_template[0])) # copy.deepcopy will make a non-shadow copy otherwise the new element in the list is just a pointer to the fist one and if anyone of them is set (in loop the last one) all will be set to the same. 
    #print(i)


#template = copy.deepcopy(engagements_template[0])

for idx, row in df.iterrows():

    #clean two day training dates
    date_str = row['training_date']
    if '-' in date_str and '/' in date_str:
        # Assume format like "26-27/7/2020" -> take first part "26/7/2020"
        start = date_str.split('-')[0] + date_str[date_str.find('/'):]
        date_str = start
    training_date = pd.to_datetime(date_str, dayfirst=True, errors='coerce').strftime('%Y-%m-%d')
    #print(training_date)
    
    #clean organization names from patners
    patners = row['partners'].strip()
    # Extract first partner
    patner1 = re.split(r'\s*[&/,]\s*', patners)[0]
    organization = patner1.strip()

    #clean notes
    notess = row['notes'].replace('\n', ' ').replace(',', ' ').replace('\"', '').replace("'", "").replace('.', '').replace('-', '').replace('  ', ' ').capitalize().strip() if pd.notna(row['notes']) else "None"
    cleaned_notes = re.sub(r'\d+', '', notess) #remove numbers from notes

    #assign df values to json values
    json_list[idx]['id'] = idx
    json_list[idx]['organization'] = organization.capitalize() if organization.lower() != 'icraf project team' else 'ICRAF'
    json_list[idx]['date_collected'] = pd.to_datetime(row['date'].replace('/', '-'), dayfirst=True, errors='coerce').strftime('%Y-%m-%d')
    json_list[idx]['engagement_date'] = training_date
    json_list[idx]['engagement_venue'] = row['training_venue'].strip().replace(',' , ' ').replace ('/', ' ').replace('  ', ' ').capitalize()
    json_list[idx]['policy_issue'] = row['training_topic'].capitalize().strip()
    json_list[idx]['engagement_type'] = row['training_type'].replace(',' , ' and ').replace('  ', ' ').capitalize().strip()
    json_list[idx]['key_participants'] = patners.replace('/', ' and ').replace('&', 'and').replace(',' , ' ').replace('  ', ' ').capitalize()
    json_list[idx]['total_participants'] = row['total_participants']
    json_list[idx]['male_participants'] = row['male_participants']
    json_list[idx]['female_participants'] = row['female_participants']
    json_list[idx]['youth_participants'] = row['youth_participants']
    json_list[idx]['comment'] = cleaned_notes
    json_list[idx]['project']['project_name'] = row['survey_name']
    json_list[idx]['subcounty']['subcounty_name'] = row['district_commune_woreda'].replace(',', ' ').replace('/', ' ').replace('  ', ' ').capitalize().strip()
    json_list[idx]['subcounty']['county']['county_name'] = row['region'].replace(',', '').replace('/', ' ').replace('  ', ' ').capitalize().strip()
    json_list[idx]['subcounty']['county']['country']['country_name'] = row['country']

len(json_list)

#save json to file
with open('/Users/victor/Documents/projects/mini_projects/regreen_cleaning/2026/ra_v1/somalia/ra_v1_engagements.json', 'w') as f:
    json.dump(json_list, f, indent=4)


# migration status
```sql
-- @ - cleaned
-- ## - added to json
+-------------------------+--------------+------+-----+-------------------+----------------+
| Field                   | Type         | Null | Key | Default           | Extra          |
+-------------------------+--------------+------+-----+-------------------+----------------+
| id                      | int(11)      | NO   | PRI | NULL              | auto_increment |
| uploaded_at             | timestamp    | NO   |     | CURRENT_TIMESTAMP |                |
| module                  | varchar(50)  | NO   |     | NULL              |                |
| enumerator_name  ## @      | varchar(100) | NO   |     | NULL              |                |
| date             ## @      | varchar(50)  | NO   |     | NULL              |                |
| survey_name      ## @      | varchar(100) | NO   |     | NULL              |                |
| country          ##       | varchar(50)  | NO   |     | NULL              |                |
| region           ## @      | varchar(50)  | NO   |     | NULL              |                |
| district_commune_woreda ## @| varchar(50)  | NO   |     | NULL              |                |
| training_topic   ## @      | varchar(100) | NO   |     | NULL              |                |
| training_type    ## @      | varchar(150) | NO   |     | NULL              |                |
| training_date    ## @      | varchar(50)  | NO   |     | NULL              |                |
| training_venue   ## @      | varchar(50)  | NO   |     | NULL              |                |
| partners         ## @      | varchar(100) | NO   |     | NULL              |                |
| total_participants ##     | int(11)      | NO   |     | NULL              |                |
| male_participants  ##     | int(11)      | NO   |     | NULL              |                |
| female_participants ##    | int(11)      | NO   |     | NULL              |                |
| youth_participants  ##    | int(11)      | NO   |     | NULL              |                |
| notes         ## @         | text         | NO   |     | NULL              |                |
+-------------------------+--------------+------+-----+-------------------+----------------+
19 rows in set (0.00 sec)
```